# Step 2 — Chunking: Build the RAG Document Database

This notebook assembles all text chunks that will go into the FAISS retrieval index.
It outputs a single `chunks/chunks.csv` file consumed by the next notebook (`rag_step3_index.ipynb`).

**Sources combined here:**
- IHC training examples (loaded from HuggingFace, `post` column)
- ISHate training examples (loaded from HuggingFace, `text` column)
- External documents: all `.txt` files found in the `documents/` folder (lexicons, definitions)

**Output CSV schema:**
```
chunk_id | source        | text                             | label
---------|---------------|----------------------------------|----------
0        | ihc_train     | "[hate] you are all subhuman"    | hate
1        | ishate_train  | "[not hate] I love this city"    | not_hate
2        | hurtlex.txt   | "derogatory term for group X"    | unknown
```

The label prefix is **baked into the `text` field** so the encoder sees it at indexing and query time.
The separate `label` column is kept for analysis and potential filtering.

## 1. Imports

In [ ]:
# TODO: imports
# - pandas
# - os, glob (for scanning documents/ folder)
# - datasets (load_dataset)
# - transformers AutoTokenizer (for token-length counting only)
# - matplotlib (for histogram in token analysis cell)
# - numpy

## 2. Load IHC Training Data

Dataset: `tasksource/implicit-hate-stg1`  
Text column: `post`  
Label column: `class` → `not_hate` maps to `not_hate`, everything else maps to `hate`

**Important:** use only the **training split** (same 90/10 split with seed=42 as in baseline_step1).
The test split must never enter the index — it is only ever used as a query at inference time.

Each example is formatted as a plain string: `"[hate] original tweet text"` or `"[not hate] original tweet text"`

In [ ]:
# TODO:
# 1. load_dataset("tasksource/implicit-hate-stg1", split="train")
# 2. train_test_split(test_size=0.10, seed=42)  → take ["train"] only
# 3. map labels: "not_hate" → "not_hate", else → "hate"
# 4. format each row: f"[{label}] {row['post']}"
# 5. build list of dicts: {"source": "ihc_train", "text": ..., "label": ...}
# 6. print number of hate / not_hate examples

## 3. Load ISHate Training Data

Dataset: `BenjaminOcampo/ISHate`  
Text column: `text`  
Label column: `hateful_layer` → `Non-HS` maps to `not_hate`, everything else maps to `hate`

ISHate has a pre-made train split — use it directly, no manual split needed.
Same string format: `"[hate] tweet"` or `"[not hate] tweet"`

In [ ]:
# TODO:
# 1. load_dataset("BenjaminOcampo/ISHate")["train"]
# 2. map labels: "Non-HS" → "not_hate", else → "hate"
# 3. format: f"[{label}] {row['text']}"
# 4. build list of dicts: {"source": "ishate_train", "text": ..., "label": ...}
# 5. print counts

## 4. Load and Chunk External Documents from `documents/`

Scan all `.txt` files in `documents/`. Each file is read and split into chunks.

**Chunking strategy:**
1. Split on blank lines (`\n\n`) to get paragraphs
2. If a paragraph exceeds `MAX_TOKENS` (target: 100 tokens), split further on `. ` (sentences)
3. Discard chunks that are empty or under 3 tokens (noise)

Token counting uses the HateBERT tokenizer (most conservative vocabulary — longest tokenization).
All external document chunks get `label="unknown"` since they are not labelled examples.

**Format for `documents/` files:**  
Put one entry per line for lexicons (word or short phrase per line).  
For typology definitions, separate categories with a blank line.

In [ ]:
# TODO:
# 1. tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT") for token counting
# 2. glob all .txt files in documents/
# 3. for each file:
#    a. read text
#    b. split on \n\n → paragraphs
#    c. for each paragraph: count tokens
#       - if <= MAX_TOKENS: keep as-is
#       - else: split on ". " and add each sentence as a separate chunk
#    d. filter chunks with < 3 tokens
#    e. add dicts: {"source": filename, "text": chunk, "label": "unknown"}
# 4. print count per file

## 5. Combine All Sources

In [ ]:
# TODO:
# 1. concatenate all lists of dicts into one list
# 2. df = pd.DataFrame(all_chunks)
# 3. df.insert(0, "chunk_id", range(len(df)))  ← row index becomes the FAISS vector index
# 4. print df.shape, df["source"].value_counts(), df["label"].value_counts()

## 6. Token Length Analysis

Verify that chunks are within the 512-token budget.
The final input to the encoder at inference time will be:
`[CLS] tweet [SEP] chunk1 [SEP] chunk2 [SEP]`

With up to 3 retrieved chunks and a tweet of ~50 tokens:
`50 + 3 × chunk_length + 5 special tokens ≤ 512`
→ chunks should be ≤ **~150 tokens** each (100 is safer).

Plot a histogram of token lengths and print 95th percentile + max.

In [ ]:
# TODO:
# 1. token_lengths = [len(tokenizer.encode(t)) for t in df["text"]]
# 2. plt.hist(token_lengths, bins=50) + labels
# 3. print: mean, median, 95th percentile, max
# 4. print: how many chunks exceed 150 tokens (flag them)

## 7. Save

Save to `chunks/chunks.csv`. This file is the only input needed by `rag_step3_index.ipynb`.

In [ ]:
# TODO:
# os.makedirs("chunks", exist_ok=True)
# df.to_csv("chunks/chunks.csv", index=False)
# print(f"Saved {len(df):,} chunks to chunks/chunks.csv")